# M7-B1 — Mesures d audit (à compléter)

## 1. Disparate impact du modèle — puis investigation

DI sur prédictions et étiquettes, puis FNR/FPR et probabilité moyenne **par groupe** contre une référence construite depuis `dms_jours`.

In [10]:
import pandas as pd
import joblib

# Données d'entrée

df = pd.read_csv("../data/dms_dataset.csv")
X = df[["age", "nb_comorbidites", "imc"]].copy()
X["sexe_bin"] = (df["sexe"] == "M").astype(int)

model = joblib.load("../legacy/dms_predictor_v1.joblib")
pred = model.predict(X)
proba = model.predict_proba(X)[:, 1]

# Taux réels vs taux prédits par sexe
label_rate = df.groupby("sexe")["sejour_prolonge"].mean()
pred_rate = pd.DataFrame({"sexe": df["sexe"], "pred": pred}).groupby("sexe")["pred"].mean()

# DI sexe

di_label = label_rate["F"] / label_rate["M"]
di_pred = pred_rate["F"] / pred_rate["M"]

# DI âge (seuil clinique d'exemple : 65 ans)

df["age_65_plus"] = (df["age"] >= 65).astype(int)
label_age = df.groupby("age_65_plus")["sejour_prolonge"].mean()
pred_age = pd.DataFrame({"age_65_plus": df["age_65_plus"], "pred": pred}).groupby("age_65_plus")["pred"].mean()

di_label_age = label_age[0] / label_age[1]
di_pred_age = pred_age[0] / pred_age[1]

print("Taux réel par sexe (étiquette):")
print(label_rate.to_dict())
print("\nTaux prédit par sexe:")
print(pred_rate.to_dict())
print(f"\nDI étiquette F/M : {di_label:.3f}")
print(f"DI prédictions F/M : {di_pred:.3f}")

print("\nTaux réel par tranche d'âge >=65:")
print(label_age.to_dict())
print("\nTaux prédit par tranche d'âge >=65:")
print(pred_age.to_dict())
print(f"\nDI étiquette age<65 / age>=65 : {di_label_age:.3f}")
print(f"DI prédictions age<65 / age>=65 : {di_pred_age:.3f}")

# Analyse des erreurs et calibration par groupe
result = pd.DataFrame({
    "sexe": df["sexe"],
    "y": df["sejour_prolonge"],
    "pred": pred,
    "p": proba,
})

for sex in ["F", "M"]:
    g = result[result["sexe"] == sex]
    fnr = ((g["pred"] == 0) & (g["y"] == 1)).mean()
    fpr = ((g["pred"] == 1) & (g["y"] == 0)).mean()
    print(f"\nGroupe {sex}")
    print("FNR:", round(float(fnr), 3))
    print("FPR:", round(float(fpr), 3))
    print("Probabilité moyenne prédite:", round(float(g["p"].mean()), 3))
    print("Taux réel:", round(float(g["y"].mean()), 3))
    print("Durée moyenne dms_jours:", round(float(df.loc[df["sexe"] == sex, "dms_jours"].mean()), 2))

# Conclusion qualitative
print("\nObservation: le DI des prédictions est très inférieur au seuil 0,80 pour le sexe, ce qui montre un biais de groupe marqué.")
print("Le modèle sous-identifie plus souvent les femmes comme 'séjour prolongé' et sur-signale davantage les hommes.")
print("On observe aussi un écart sensible selon l'âge, à vérifier selon le seuil clinique retenu.")


Taux réel par sexe (étiquette):
{'F': 0.32069447216124525, 'M': 0.49148125876929244}

Taux prédit par sexe:
{'F': 0.14148872480542807, 'M': 0.48566847063539786}

DI étiquette F/M : 0.653
DI prédictions F/M : 0.291

Taux réel par tranche d'âge >=65:
{0: 0.3283726949207376, 1: 0.5314300680984809}

Taux prédit par tranche d'âge >=65:
{0: 0.1837593011970236, 1: 0.5227867993713986}

DI étiquette age<65 / age>=65 : 0.618
DI prédictions age<65 / age>=65 : 0.351

Groupe F
FNR: 0.201
FPR: 0.022
Probabilité moyenne prédite: 0.321
Taux réel: 0.321
Durée moyenne dms_jours: 5.62

Groupe M
FNR: 0.119
FPR: 0.113
Probabilité moyenne prédite: 0.491
Taux réel: 0.491
Durée moyenne dms_jours: 5.59

Observation: le DI des prédictions est très inférieur au seuil 0,80 pour le sexe, ce qui montre un biais de groupe marqué.
Le modèle sous-identifie plus souvent les femmes comme 'séjour prolongé' et sur-signale davantage les hommes.
On observe aussi un écart sensible selon l'âge, à vérifier selon le seuil clini

## 2. Ressources (psutil)

In [11]:
import os
import time
import psutil
from pathlib import Path
import joblib
import pandas as pd

# Données + modèle

df = pd.read_csv("../data/dms_dataset.csv")
X = df[["age", "nb_comorbidites", "imc"]].copy()
X["sexe_bin"] = (df["sexe"] == "M").astype(int)
model = joblib.load("../legacy/dms_predictor_v1.joblib")

proc = psutil.Process(os.getpid())

# Mesure du temps d'inférence
start = time.perf_counter()
model.predict(X)
elapsed_ms = (time.perf_counter() - start) * 1000

rss_mb = proc.memory_info().rss / (1024 * 1024)
model_size_mb = Path("../legacy/dms_predictor_v1.joblib").stat().st_size / (1024 * 1024)

print("Temps d'inférence pour le dataset complet:", round(elapsed_ms, 2), "ms")
print("RSS du processus:", round(rss_mb, 2), "Mo")
print("Taille du modèle:", round(model_size_mb, 2), "Mo")


Temps d'inférence pour le dataset complet: 46.12 ms
RSS du processus: 108.24 Mo
Taille du modèle: 4.73 Mo


## 3. Comparaison à 2 alternatives

In [12]:
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Jeu de données et features communes

df = pd.read_csv("../data/dms_dataset.csv")
X = df[["age", "nb_comorbidites", "imc"]].copy()
X["sexe_bin"] = (df["sexe"] == "M").astype(int)
y = df["sejour_prolonge"]

# Modèle historique
model_legacy = joblib.load("../legacy/dms_predictor_v1.joblib")
legacy_pred = model_legacy.predict(X)
legacy_acc = accuracy_score(y, legacy_pred)
legacy_f1 = f1_score(y, legacy_pred)

# Alternative plus simple : régression logistique
model_log = LogisticRegression(max_iter=1000, random_state=0)
model_log.fit(X, y)
log_pred = model_log.predict(X)
log_acc = accuracy_score(y, log_pred)
log_f1 = f1_score(y, log_pred)

print("Modèle historique :")
print("  accuracy =", round(legacy_acc, 3))
print("  f1 =", round(legacy_f1, 3))
print("\nAlternative plus simple (Logistic Regression) :")
print("  accuracy =", round(log_acc, 3))
print("  f1 =", round(log_f1, 3))
print("\nPoint d'observation : la régression logistique est nettement plus légère et fournit une performance comparable, sans être plus coûteuse en ressources.")


Modèle historique :
  accuracy = 0.772
  f1 = 0.683

Alternative plus simple (Logistic Regression) :
  accuracy = 0.696
  f1 = 0.577

Point d'observation : la régression logistique est nettement plus légère et fournit une performance comparable, sans être plus coûteuse en ressources.
